# streamlit: aplicaciones web de ciencia de datos

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gf0657-programacionsig/2026-ii/blob/main/contenidos/v-desarrollo-aplicaciones-interactivas/22-streamlit-aplicaciones-web.ipynb)

## Trabajo previo

### Lecturas

Antes de la clase, revise las siguientes lecturas, en inglés. La página de conceptos básicos de la documentación oficial explica, en pocas pantallas, el modelo de ejecución de streamlit, que es lo único realmente nuevo de este cuaderno; los dos capítulos del libro de Richards recorren la instalación, la primera aplicación y la carga de datos con el ritmo de un tutorial.

Snowflake Inc. (s. f.). Basic concepts of Streamlit. En *Streamlit documentation*. Recuperado el 1 de noviembre de 2026, de https://docs.streamlit.io/get-started/fundamentals/main-concepts
\
\
Richards, T. (2023). *Streamlit for data science: Create interactive data apps in Python* (2.ª ed., capítulos 1 y 2). Packt Publishing.

### Otros recursos

La [referencia de la API](https://docs.streamlit.io/develop/api-reference) de streamlit documenta cada función con un ejemplo ejecutable, y la [galería](https://streamlit.io/gallery) muestra aplicaciones publicadas con su código. La documentación de [Streamlit Community Cloud](https://docs.streamlit.io/deploy/streamlit-community-cloud) describe el servicio gratuito en que se publicará la aplicación del proyecto, y el repositorio de [streamlit-folium](https://github.com/randyzwitch/streamlit-folium), el componente que lleva los mapas de folium a streamlit.

## Introducción

Todo lo que el curso produjo hasta ahora vive en cuadernos de notas: tablas de pandas, gráficos de plotly, mapas de folium y leafmap. Un cuaderno es un documento para quien programa; el **proyecto final** pide algo distinto: una **aplicación web interactiva**, publicada en Internet, que cualquier persona pueda usar sin ver el código. [streamlit](https://streamlit.io/) es un marco de trabajo de código abierto, creado en 2019 y mantenido por Snowflake, que convierte un programa de Python en una página web: cada llamada a una función de la biblioteca (`st.title()`, `st.dataframe()`, `st.slider()`) dibuja un elemento en la página, y los elementos de entrada devuelven lo que la persona usuaria elige, como variables normales de Python. No hace falta HTML, CSS ni JavaScript. Otros marcos de trabajo cumplen el mismo propósito ([Dash](https://dash.plotly.com/), [Panel](https://panel.holoviz.org/), [Shiny para Python](https://shiny.posit.co/py/), [Gradio](https://www.gradio.app/)); streamlit es el más simple de aprender y el más usado para prototipos de ciencia de datos, y es el que pide el programa del curso.

La idea central cabe en una frase, y conviene retenerla porque explica todo lo demás: **una aplicación de streamlit es un programa de Python que se ejecuta de arriba abajo cada vez que la persona usuaria toca un control**. No hay eventos ni funciones de respuesta que programar: se escribe el programa como un guion lineal (cargar los datos, leer los controles, filtrar, mostrar) y streamlit lo vuelve a ejecutar completo con los valores nuevos. Lo que cuesta tiempo, como leer los datos, se guarda en una **caché** para no repetirlo.

Este cuaderno es distinto de los anteriores en un aspecto práctico: una aplicación no corre dentro del cuaderno, sino como un programa aparte (`streamlit run archivo.py`) que se ve en el navegador. Por eso cada ejemplo se escribe en un archivo de la carpeta `apps/` (la celda empieza con `%%writefile`, que guarda su contenido en el archivo en lugar de ejecutarlo), se prueba desde el cuaderno con `AppTest`, la herramienta de pruebas de streamlit, que ejecuta la aplicación sin navegador y permite examinar lo que produce, y aparece en el sitio del curso como una captura de pantalla. Para ver y usar las aplicaciones hay que ejecutarlas en la computadora propia: el ambiente `geopython` del curso incluye streamlit; el componente de mapas se agrega con la actualización del ambiente que se indica más adelante. En Colab las celdas se ejecutan y las pruebas funcionan, pero el navegador no puede abrir la aplicación, de modo que este cuaderno se trabaja mejor en Visual Studio Code, con una terminal abierta junto al cuaderno. El archivo `.ipynb` puede descargarse con el botón de descarga (ícono de flecha hacia abajo) en la parte superior de la página del cuaderno, en el sitio web del curso.

In [1]:
# Instalación solo en Google Colab (en un ambiente local, esta celda no hace nada)
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    %pip install -q streamlit streamlit-folium

In [2]:
import os

import pandas as pd
import streamlit as st
from streamlit.testing.v1 import AppTest

# Las aplicaciones de ejemplo se escriben en esta carpeta, junto al cuaderno
os.makedirs("apps", exist_ok=True)

# Dirección de los datos en el repositorio del curso
DATOS = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos"

print("streamlit", st.__version__)

streamlit 1.61.1


La función auxiliar siguiente se usa en todo el cuaderno: ejecuta una aplicación con `AppTest` y resume lo que produjo (título, controles, tablas, gráficos, mapas y errores). Es el equivalente de las celdas de verificación de los cuadernos anteriores, y la misma herramienta sirve para probar la aplicación del proyecto sin abrir el navegador:

In [3]:
def probar(archivo, **valores):
    # Ejecuta una aplicación de streamlit sin navegador y resume lo que produce.
    # Los argumentos con nombre fijan el valor de un control antes de ejecutar, por su etiqueta:
    # probar("apps/03-controles.py", Provincia="Heredia"). La ruta se convierte en absoluta porque
    # AppTest la interpreta respecto del archivo que lo llama, no del directorio de trabajo.
    app = AppTest.from_file(os.path.abspath(archivo), default_timeout=120).run()
    for etiqueta, valor in valores.items():
        for control in app.get("slider") + app.get("selectbox") + app.get("multiselect") + app.get("checkbox") + app.get("radio") + app.get("text_input"):
            if control.label == etiqueta:
                control.set_value(valor)
        app.run()
    cuenta = lambda tipo: len(app.get(tipo))
    print(f"{archivo}: ", end="")
    print(f"título: {app.title[0].value!r} | " if app.title else "sin título | ", end="")
    print(f"controles: {cuenta('slider') + cuenta('selectbox') + cuenta('multiselect') + cuenta('checkbox') + cuenta('radio') + cuenta('text_input')} | "
          f"métricas: {cuenta('metric')} | tablas: {cuenta('dataframe')} | gráficos: {cuenta('plotly_chart')} | "
          f"mapas: {cuenta('deck_gl_json_chart') + cuenta('component_instance')} | errores: {len(app.exception)}")
    for error in app.exception:
        print("  ERROR:", error.value.splitlines()[0])
    return app

*Ejercicios de esta sección: 1, en la sección de ejercicios al final del cuaderno.*

## Una aplicación es un programa

La aplicación más pequeña posible tiene tres líneas: importar streamlit (por convención, como `st`), un título y un texto. `%%writefile` guarda la celda en el archivo `apps/01-hola.py` en lugar de ejecutarla:

In [4]:
%%writefile apps/01-hola.py
import streamlit as st

st.title("Mi primera aplicación")
st.write("Esta página la produce un programa de Python de unas pocas líneas.")

nombre = st.text_input("Escriba su nombre", "estudiante")
st.write(f"Hola, {nombre}. Cada vez que cambia el texto, el programa completo se ejecuta de nuevo.")

Writing apps/01-hola.py


Para verla, se ejecuta desde una terminal, con el ambiente del curso activado y desde la carpeta del cuaderno:

```bash
streamlit run apps/01-hola.py
```

streamlit arranca un servidor web local, imprime su dirección (`http://localhost:8501`) y abre el navegador. La página se actualiza cuando el archivo cambia (el menú de la esquina superior derecha ofrece *Rerun* y, mejor, *Always rerun*, que recarga la página cada vez que se guarda el archivo: así se desarrolla, con el editor y el navegador lado a lado). Se detiene con Ctrl+C en la terminal. La figura 1 muestra el resultado.

<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/v-desarrollo-aplicaciones-interactivas/img/streamlit-01-hola.png" alt="Página de streamlit con un título, un párrafo, un cuadro de texto y un saludo" width="900">
  <figcaption><strong>Figura 1</strong>. La primera aplicación en el navegador: el título, el texto, el cuadro de texto y el saludo que cambia al escribir. Fuente: elaboración propia.</figcaption>
</figure>

Tres cosas que muestra ese programa. Primera, cada función `st.*` **dibuja** algo en la página, en el orden en que se llama: la página es la secuencia de llamadas. `st.write()` es la función comodín: recibe texto (con Markdown), números, DataFrames, figuras de matplotlib o plotly, diccionarios, y muestra cada cosa de la forma apropiada; las demás funciones de salida son versiones especializadas (`st.title()`, `st.header()`, `st.markdown()`, `st.dataframe()`, `st.plotly_chart()`...). Segunda, un **control** como `st.text_input()` dibuja el cuadro de texto **y retorna su valor**: `nombre` es una cadena normal de Python que se usa como cualquier otra. Tercera, y es la clave: al escribir en el cuadro, streamlit vuelve a ejecutar **todo el programa** desde la primera línea, ahora con el valor nuevo, y redibuja la página. No hay nada que "escuche" el cambio: el programa es la respuesta.

La celda siguiente ejecuta la aplicación sin navegador y resume lo que produce; es lo que hará el cuaderno con cada ejemplo:

In [5]:
app = probar("apps/01-hola.py")

apps/01-hola.py: título: 'Mi primera aplicación' | controles: 1 | métricas: 0 | tablas: 0 | gráficos: 0 | mapas: 0 | errores: 0


Al ejecutar una aplicación de esta forma, streamlit escribe en la terminal (o debajo de la celda) algunas líneas de registro, como *missing ScriptRunContext*, que se pueden ignorar; en la versión publicada del cuaderno se omiten. `AppTest` da acceso a cada elemento de la página por su tipo: el título, los controles, las tablas. Y permite **simular** a la persona usuaria: fijar el valor de un control y volver a ejecutar, para comprobar que la aplicación responde como se espera. Con el nombre cambiado, el saludo cambia:

In [6]:
app.text_input[0].set_value("Ana").run()
print(app.markdown[-1].value)

Hola, Ana. Cada vez que cambia el texto, el programa completo se ejecuta de nuevo.


*Ejercicios de esta sección: 2, en la sección de ejercicios al final del cuaderno.*

## Datos y caché

Una aplicación del proyecto carga sus datos desde una URL, como los cuadernos de las tareas: así corre igual en la computadora propia y en el servidor que la publique. Pero hay una diferencia con el cuaderno: como el programa se ejecuta de nuevo con cada interacción, una lectura de varios segundos se repetiría cada vez que se mueve un control. La solución es el **decorador** `@st.cache_data`: la primera vez que la función se llama, streamlit ejecuta el cuerpo y guarda el resultado; en las llamadas siguientes con los mismos argumentos, retorna la copia guardada sin ejecutar nada. Es la regla para toda función que lea o calcule algo costoso. (Su pariente `@st.cache_resource` guarda objetos que no deben copiarse, como una conexión a una base de datos; para datos, siempre `cache_data`.)

La aplicación siguiente carga los cantones del IGN con la población del INEC, como en los cuadernos de folium y leafmap, y los presenta con tres elementos de salida nuevos: `st.metric()`, un indicador grande con etiqueta; `st.dataframe()`, la tabla interactiva (se ordena por columna al hacer clic en el encabezado, se buscan valores y se descarga desde su menú), con `column_config` para los encabezados y el formato de las columnas; y `st.download_button()`, que entrega un archivo. `st.columns()` reparte la página en columnas y cada una recibe elementos como si fuera la página:

In [7]:
%%writefile apps/02-datos.py
import geopandas as gpd
import pandas as pd
import streamlit as st

DATOS = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos"


# Los datos se leen una sola vez: streamlit guarda el resultado y lo reutiliza en cada ejecución
@st.cache_data
def cargar_cantones():
    cantones = gpd.read_file(f"{DATOS}/ign/cantones.gpkg").to_crs(4326)
    poblacion = pd.read_csv(f"{DATOS}/inec/cantones-2022.csv")
    return cantones.merge(poblacion[["codigo_canton", "poblacion", "area_km2", "densidad"]], on="codigo_canton", how="left")


st.title("Cantones de Costa Rica")
st.markdown("División territorial del [IGN](https://www.snitcr.go.cr/) y estimación de población 2022 del [INEC](https://inec.cr/).")

cantones = cargar_cantones()

# Indicadores en tres columnas
col1, col2, col3 = st.columns(3)
col1.metric("Cantones", len(cantones))
col2.metric("Población 2022", f"{cantones['poblacion'].sum():,.0f}".replace(",", " "))
col3.metric("Área (km²)", f"{cantones['area_km2'].sum():,.0f}".replace(",", " "))

# Tabla interactiva: sin la columna de geometría, que no es una columna de datos
tabla = cantones.drop(columns="geometry").sort_values("poblacion", ascending=False)
st.dataframe(
    tabla,
    hide_index=True,
    width="stretch",
    column_config={
        "codigo_provincia": None,  # columna oculta
        "codigo_canton": "Código",
        "provincia": "Provincia",
        "canton": "Cantón",
        "poblacion": st.column_config.NumberColumn("Población", format="localized"),
        "area_km2": st.column_config.NumberColumn("Área (km²)", format="%.1f"),
        "densidad": st.column_config.NumberColumn("Densidad (hab/km²)", format="%.1f"),
    },
)

st.download_button("Descargar la tabla (CSV)", tabla.to_csv(index=False).encode("utf-8"), "cantones-2022.csv", "text/csv")

Writing apps/02-datos.py


In [8]:
app = probar("apps/02-datos.py")
print(app.metric[0].label, app.metric[0].value, "|", app.metric[1].label, app.metric[1].value)

apps/02-datos.py: título: 'Cantones de Costa Rica' | controles: 0 | métricas: 3 | tablas: 1 | gráficos: 0 | mapas: 0 | errores: 0
Cantones 84 | Población 2022 5 044 197


<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/v-desarrollo-aplicaciones-interactivas/img/streamlit-02-datos.png" alt="Página con tres indicadores, una tabla interactiva de cantones y un botón de descarga" width="900">
  <figcaption><strong>Figura 2</strong>. Indicadores, tabla interactiva y botón de descarga. La tabla se ordena al hacer clic en un encabezado y tiene su propio menú de búsqueda y descarga. Fuente: elaboración propia.</figcaption>
</figure>

Dos detalles de la tabla. La columna `geometry` se quita antes de mostrarla: `st.dataframe()` convierte el DataFrame al formato Arrow para enviarlo al navegador, y las geometrías no tienen representación en él (la clínica muestra qué pasa si se deja). Y `column_config` recibe, por columna, un nombre para el encabezado, `None` para ocultarla o un objeto `st.column_config.NumberColumn()` con el formato: `"localized"` usa los separadores de miles del navegador y `"%.1f"` es el formato de Python.

*Ejercicios de esta sección: 3 y 4, en la sección de ejercicios al final del cuaderno.*

## Controles y disposición

Los controles (*widgets*) son la razón de ser de una aplicación: cada uno dibuja un elemento de entrada y retorna su valor. Los más usados para filtrar datos son `st.selectbox()` (una opción de una lista), `st.multiselect()` (varias), `st.slider()` (un número o un rango), `st.checkbox()` y `st.radio()`. Como retornan valores de Python, el filtrado es pandas: la aplicación no tiene lógica nueva, solo el programa de siempre con los valores que la persona eligió.

La **barra lateral**, `st.sidebar`, es el lugar convencional para los controles: cualquier función `st.*` tiene su versión `st.sidebar.*`, que dibuja en la barra en lugar de la página. Y tres contenedores organizan la página: `st.columns()`, ya vista; `st.tabs()`, pestañas; y `st.expander()`, una sección plegada. Un `with` dirige a cada contenedor los elementos que se dibujan dentro del bloque:

In [9]:
%%writefile apps/03-controles.py
import geopandas as gpd
import pandas as pd
import streamlit as st

DATOS = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos"


@st.cache_data
def cargar_cantones():
    cantones = gpd.read_file(f"{DATOS}/ign/cantones.gpkg").to_crs(4326)
    poblacion = pd.read_csv(f"{DATOS}/inec/cantones-2022.csv")
    return cantones.merge(poblacion[["codigo_canton", "poblacion", "area_km2", "densidad"]], on="codigo_canton", how="left")


st.title("Cantones de Costa Rica")
cantones = cargar_cantones()

# Controles en la barra lateral: cada uno retorna el valor elegido
st.sidebar.header("Filtros")
provincias = ["Todas"] + sorted(cantones["provincia"].unique())
provincia = st.sidebar.selectbox("Provincia", provincias)
poblacion_minima = st.sidebar.slider("Población mínima", 0, 100_000, 0, step=5_000)
ordenar_por = st.sidebar.radio("Ordenar por", ["poblacion", "area_km2", "densidad"],
                               format_func=lambda v: {"poblacion": "Población", "area_km2": "Área", "densidad": "Densidad"}[v])
descendente = st.sidebar.checkbox("De mayor a menor", value=True)

# El filtrado es pandas con los valores de los controles
seleccion = cantones if provincia == "Todas" else cantones[cantones["provincia"] == provincia]
seleccion = seleccion[seleccion["poblacion"].fillna(0) >= poblacion_minima]  # fillna: los cantones sin dato no se pierden
seleccion = seleccion.sort_values(ordenar_por, ascending=not descendente)

col1, col2 = st.columns(2)
col1.metric("Cantones seleccionados", len(seleccion))
col2.metric("Población", f"{seleccion['poblacion'].sum():,.0f}".replace(",", " "))

# Pestañas: cada bloque with dibuja dentro de su pestaña
pestana_tabla, pestana_resumen = st.tabs(["Tabla", "Resumen por provincia"])
with pestana_tabla:
    st.dataframe(seleccion.drop(columns="geometry")[["provincia", "canton", "poblacion", "area_km2", "densidad"]],
                 hide_index=True, width="stretch")
with pestana_resumen:
    resumen = seleccion.groupby("provincia")[["poblacion", "area_km2"]].sum().round(0)
    st.dataframe(resumen, width="stretch")

with st.expander("Acerca de los datos"):
    st.markdown("Cantones del IGN (84, simplificados a 100 m) y estimación de población 2022 del INEC (82 cantones: "
                "Monteverde y Puerto Jiménez quedan sin datos).")

Writing apps/03-controles.py


In [10]:
app = probar("apps/03-controles.py")
app = probar("apps/03-controles.py", Provincia="Heredia")
print("Cantones de Heredia con el filtro:", app.metric[0].value)

apps/03-controles.py: título: 'Cantones de Costa Rica' | controles: 4 | métricas: 2 | tablas: 2 | gráficos: 0 | mapas: 0 | errores: 0


apps/03-controles.py: título: 'Cantones de Costa Rica' | controles: 4 | métricas: 2 | tablas: 2 | gráficos: 0 | mapas: 0 | errores: 0
Cantones de Heredia con el filtro: 10


<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/v-desarrollo-aplicaciones-interactivas/img/streamlit-03-controles.png" alt="Página con una barra lateral de filtros, dos indicadores, pestañas y una sección plegable" width="900">
  <figcaption><strong>Figura 3</strong>. Controles en la barra lateral (lista desplegable, control deslizante, botones de opción y casilla), indicadores, pestañas y una sección plegable. Con Heredia elegida, todo lo demás cambia. Fuente: elaboración propia.</figcaption>
</figure>

Dos observaciones. `format_func` cambia cómo se muestra cada opción sin cambiar el valor que se retorna: la persona ve "Población", el programa recibe `"poblacion"`, el nombre de la columna. Y como el programa se ejecuta completo cada vez, el orden de las llamadas es el orden de la página, pero el orden de las **variables** es el del programa: un control debe leerse antes de usarse, y un valor que se calcula arriba no sabe nada de un control que está más abajo.

Entre una ejecución y otra, las variables desaparecen: el programa empieza de cero. Cuando algo debe recordarse (un contador de clics, la última selección en un mapa), se guarda en `st.session_state`, un diccionario que sobrevive a las ejecuciones de la misma sesión del navegador. Las aplicaciones del proyecto rara vez lo necesitan; la sección de mapas lo usa una vez.

*Ejercicios de esta sección: 5 y 6, en la sección de ejercicios al final del cuaderno.*

## Gráficos

Un gráfico de plotly se muestra con `st.plotly_chart()`, con toda su interactividad (acercamiento, etiquetas al pasar el cursor, descarga como imagen). La figura se construye exactamente como en el cuaderno de plotly: el único cambio es que `fig.show()` o `display(fig)` se sustituyen por `st.plotly_chart(fig, width="stretch")`. Lo mismo vale para matplotlib (`st.pyplot(fig)`), y streamlit trae además gráficos propios de una línea (`st.bar_chart()`, `st.line_chart()`, `st.scatter_chart()`) para una exploración rápida, sin títulos ni etiquetas de ejes. Para el proyecto se usa plotly, que permite documentar el gráfico como pide el curso. Un gráfico de barras cuyo número de cantones lo decide un control:

In [11]:
%%writefile apps/04-graficos.py
import geopandas as gpd
import pandas as pd
import plotly.express as px
import streamlit as st

DATOS = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos"


@st.cache_data
def cargar_cantones():
    cantones = gpd.read_file(f"{DATOS}/ign/cantones.gpkg").to_crs(4326)
    poblacion = pd.read_csv(f"{DATOS}/inec/cantones-2022.csv")
    return cantones.merge(poblacion[["codigo_canton", "poblacion", "area_km2", "densidad"]], on="codigo_canton", how="left")


st.title("Población de los cantones, 2022")
cantones = cargar_cantones()

cuantos = st.slider("Número de cantones", 5, 30, 10)
mayores = cantones.nlargest(cuantos, "poblacion")

# El gráfico se construye como en el cuaderno de plotly; st.plotly_chart() lo dibuja en la página
grafico = px.bar(
    mayores, x="canton", y="poblacion", color="provincia",
    labels={"canton": "Cantón", "poblacion": "Población (habitantes)", "provincia": "Provincia"},
    title=f"Los {cuantos} cantones más poblados de Costa Rica, 2022",
)
st.plotly_chart(grafico, width="stretch")

# Un segundo gráfico, de dispersión, con escala logarítmica y etiquetas al pasar el cursor
dispersion = px.scatter(
    cantones, x="area_km2", y="poblacion", hover_name="canton", color="provincia", log_x=True, log_y=True,
    labels={"area_km2": "Área (km²)", "poblacion": "Población (habitantes)", "provincia": "Provincia"},
    title="Área y población de los cantones, 2022",
)
st.plotly_chart(dispersion, width="stretch")

Writing apps/04-graficos.py


In [12]:
app = probar("apps/04-graficos.py")

apps/04-graficos.py: título: 'Población de los cantones, 2022' | controles: 1 | métricas: 0 | tablas: 0 | gráficos: 2 | mapas: 0 | errores: 0


<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/v-desarrollo-aplicaciones-interactivas/img/streamlit-04-graficos.png" alt="Página con un control deslizante y un gráfico de barras de plotly coloreado por provincia" width="900">
  <figcaption><strong>Figura 4</strong>. Un gráfico de plotly en streamlit, con el número de barras decidido por un control deslizante. El gráfico conserva su interactividad: acercamiento, etiquetas y descarga como imagen. Fuente: elaboración propia.</figcaption>
</figure>

*Ejercicios de esta sección: 7, en la sección de ejercicios al final del cuaderno.*

## Mapas

streamlit trae un mapa de puntos propio, `st.map()`, que recibe un DataFrame con columnas de latitud y longitud y dibuja los puntos sobre un mapa base, con tamaño y color opcionales. Sirve para una vista rápida; no admite polígonos, leyendas ni etiquetas emergentes. Los aeródromos del IGN, con las coordenadas como columnas:

In [13]:
%%writefile apps/05-mapa-puntos.py
import geopandas as gpd
import streamlit as st

DATOS = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos"


@st.cache_data
def cargar_aerodromos():
    aerodromos = gpd.read_file(f"{DATOS}/ign/aerodromos.gpkg").to_crs(4326)
    aerodromos["longitud"] = aerodromos.geometry.x
    aerodromos["latitud"] = aerodromos.geometry.y
    return aerodromos.drop(columns="geometry")  # un DataFrame común, con las coordenadas como columnas


st.title("Aeródromos de Costa Rica")
aerodromos = cargar_aerodromos()

st.map(aerodromos, latitude="latitud", longitude="longitud", size=1500, color="#D97B1B", zoom=7)
st.caption(f"{len(aerodromos)} aeródromos. Fuente: IGN.")

Writing apps/05-mapa-puntos.py


In [14]:
app = probar("apps/05-mapa-puntos.py")

apps/05-mapa-puntos.py: título: 'Aeródromos de Costa Rica' | controles: 0 | métricas: 0 | tablas: 0 | gráficos: 0 | mapas: 1 | errores: 0


<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/v-desarrollo-aplicaciones-interactivas/img/streamlit-05-mapa-puntos.png" alt="Mapa de streamlit con los aeródromos de Costa Rica como puntos naranja sobre un mapa base" width="900">
  <figcaption><strong>Figura 5</strong>. El mapa propio de streamlit, `st.map()`: puntos de un DataFrame sobre un mapa base, sin leyenda ni etiquetas. Fuente: elaboración propia.</figcaption>
</figure>

### Mapas de folium con streamlit-folium

Para los mapas del curso, con polígonos, coropletas, varias capas, leyenda y etiquetas emergentes, se usa folium, y el componente [streamlit-folium](https://github.com/randyzwitch/streamlit-folium) lo incrusta en la página. No viene con streamlit y se agregó al ambiente del curso en noviembre, por lo que un ambiente `geopython` creado antes no lo tiene. La manera de instalarlo depende de dónde se ejecute la aplicación:

- **En la computadora local (ambiente `geopython`)**: descargue la versión actual de [environment.yml](https://github.com/gf0657-programacionsig/2026-ii/blob/main/environment.yml) y, en una terminal ubicada en la carpeta donde lo guardó, actualice el ambiente, como indica la [guía de Miniconda](https://gf0657-programacionsig.github.io/2026-ii/miniconda/):

```shell
conda env update -f environment.yml
```

- Si no tiene el archivo a mano, puede instalar solo este paquete, con la misma versión del ambiente del curso:

```shell
conda activate geopython
conda install -c conda-forge streamlit-folium=0.27.4
```

- **En Google Colab**: no hay que hacer nada, porque la celda de instalación del inicio de este cuaderno lo instala.
- **En Streamlit Community Cloud**: no se instala a mano; basta con la línea `streamlit-folium==0.27.4` en `requirements.txt`, como se explica en la sección de publicación.

Para comprobar la instalación local, `conda list streamlit-folium` debe mostrar la versión 0.27.4. El paquete se llama `streamlit-folium`, con guion, pero se importa con guion bajo: `from streamlit_folium import st_folium`.

Su función `st_folium()` recibe un `folium.Map` (el que construye folium, leafmap con el motor de folium o `explore()` de geopandas) y lo dibuja. Y hace algo más: es un componente **bidireccional**, que retorna un diccionario con lo que la persona hace en el mapa, como el último clic (`last_clicked`, con `lat` y `lng`), el último objeto en que hizo clic y la extensión visible (`bounds`). Con `returned_objects` se indica cuáles de esos datos interesan: la aplicación se vuelve a ejecutar solo cuando cambia uno de ellos, y no con cada movimiento del mapa.

Con las coordenadas del clic y geopandas, la aplicación averigua en qué cantón se hizo clic y muestra sus datos: el patrón de consulta de la mayoría de los mapas interactivos del proyecto. El cantón elegido se guarda en `st.session_state` para que no se pierda si el mapa se redibuja sin un clic nuevo. La primera línea, `st.set_page_config(layout="wide")`, cambia la disposición de la página, que por defecto es una columna angosta centrada, a todo el ancho del navegador, lo que los mapas agradecen; se explica en la sección siguiente:

In [15]:
%%writefile apps/06-mapa-folium.py
import copy

import geopandas as gpd
import pandas as pd
import streamlit as st
from shapely.geometry import Point
from streamlit_folium import st_folium

DATOS = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos"


@st.cache_data
def cargar_cantones():
    cantones = gpd.read_file(f"{DATOS}/ign/cantones.gpkg").to_crs(4326)
    poblacion = pd.read_csv(f"{DATOS}/inec/cantones-2022.csv")
    return cantones.merge(poblacion[["codigo_canton", "poblacion", "area_km2", "densidad"]], on="codigo_canton", how="left")


st.set_page_config(layout="wide")  # disposición ancha: el mapa necesita espacio
st.title("Densidad de población por cantón, 2022")
cantones = cargar_cantones()

# El mapa de coropletas, como en el cuaderno de geopandas I; es un folium.Map. Se construye una
# sola vez (cache_resource, porque un mapa de folium no se puede serializar) y cada ejecución
# trabaja con una copia idéntica: así streamlit-folium reconoce el mismo mapa en cada ejecución
# y conserva la vista (centro y acercamiento) con cada clic, en lugar de dibujarlo desde cero
@st.cache_resource
def crear_mapa():
    return cargar_cantones().explore(
        column="densidad", cmap="YlOrRd", scheme="Quantiles", k=5,
        location=[9.75, -84.15], zoom_start=8,  # centro y acercamiento fijos (sin la Isla del Coco)
        tooltip=["canton", "provincia", "densidad"], tiles="OpenStreetMap",
        style_kwds={"color": "white", "weight": 0.5}, missing_kwds={"color": "#E6E6E6"},
        legend_kwds={"caption": "Densidad de población, 2022 (hab/km²)"},
    )


mapa = copy.deepcopy(crear_mapa())

col_mapa, col_datos = st.columns([2, 1])
with col_mapa:
    # st_folium() dibuja el mapa y retorna lo que la persona hace en él
    salida = st_folium(mapa, height=500, use_container_width=True, returned_objects=["last_clicked"])

# Con el clic, se busca el cantón que contiene el punto y se recuerda en la sesión
if salida and salida.get("last_clicked"):
    punto = Point(salida["last_clicked"]["lng"], salida["last_clicked"]["lat"])
    encontrado = cantones[cantones.contains(punto)]
    if not encontrado.empty:
        st.session_state["canton"] = encontrado.iloc[0]

with col_datos:
    st.subheader("Cantón")
    if "canton" in st.session_state:
        canton = st.session_state["canton"]
        st.metric(canton["canton"], f"{canton['poblacion']:,.0f} hab.".replace(",", " ") if pd.notna(canton["poblacion"]) else "sin datos")
        st.write(f"Provincia: {canton['provincia']}")
        st.write(f"Área: {canton['area_km2']:,.1f} km²".replace(",", " ") if pd.notna(canton["area_km2"]) else "Área: sin datos")
        st.write(f"Densidad: {canton['densidad']:,.1f} hab/km²".replace(",", " ") if pd.notna(canton["densidad"]) else "Densidad: sin datos")
    else:
        st.info("Haga clic en un cantón del mapa.")

Writing apps/06-mapa-folium.py


In [16]:
app = probar("apps/06-mapa-folium.py")

apps/06-mapa-folium.py: título: 'Densidad de población por cantón, 2022' | controles: 0 | métricas: 0 | tablas: 0 | gráficos: 0 | mapas: 1 | errores: 0


<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/v-desarrollo-aplicaciones-interactivas/img/streamlit-06-mapa-folium.png" alt="Mapa de coropletas de folium dentro de streamlit, con los datos del cantón en que se hizo clic a la derecha" width="900">
  <figcaption><strong>Figura 6</strong>. Un mapa de folium incrustado con `st_folium()`. El clic sobre San Isidro de Heredia volvió a Python como coordenadas, geopandas encontró el cantón y la columna derecha muestra sus datos. Fuente: elaboración propia.</figcaption>
</figure>

`AppTest` no puede hacer clic en el mapa (el componente corre en el navegador), por eso la prueba muestra el mensaje inicial; la figura 6 muestra el resultado de un clic sobre San Isidro de Heredia. Si la aplicación no necesita responder al mapa, `returned_objects=[]` lo convierte en un mapa de solo lectura, más ligero.

Un detalle técnico de esta aplicación merece explicación, porque afecta a cualquier mapa con clics. Como el programa se ejecuta de nuevo con cada clic, un `explore()` suelto construiría un mapa nuevo cada vez, con identificadores internos distintos, y streamlit-folium lo dibujaría desde cero: la vista volvería al centro y al acercamiento iniciales con cada clic, lo que resulta exasperante. Por eso el mapa se construye en una función con `@st.cache_resource` (un `folium.Map` no se puede serializar, de modo que `cache_data` no sirve) y cada ejecución recibe una copia con `copy.deepcopy()`, idéntica al original, incluidos los identificadores. streamlit-folium reconoce el mismo mapa y conserva la vista. La copia es necesaria porque dibujar el mapa lo modifica, y no debe modificarse el que está en la caché. leafmap ofrece lo mismo con su método `to_streamlit()`, que usa este componente por dentro, y folium puro, geopandas y leafmap con el motor de folium producen el mismo objeto `folium.Map`: todo lo construido en la sección IV se reutiliza sin cambios.

*Ejercicios de esta sección: 8 y 9, en la sección de ejercicios al final del cuaderno.*

## La aplicación del proyecto

Con esos elementos se construye la aplicación del proyecto final: tablas, gráficos y mapas del tema de cada estudiante, con controles que los filtren, publicada en Internet. Es una sola aplicación, en un repositorio propio con esta estructura mínima:

```text
mi-proyecto/
├── app.py             # la aplicación
├── requirements.txt   # las bibliotecas que necesita, con sus versiones
├── README.md          # qué es, enlace a la aplicación publicada, fuentes de los datos
└── datos/             # opcional: solo si los datos no se cargan desde una URL externa
```

Y con estas convenciones, que resumen lo visto:

- `st.set_page_config()` es la primera llamada a streamlit: título de la pestaña del navegador, ícono y disposición (`layout="wide"` usa todo el ancho, lo que conviene a los mapas).
- Los datos se cargan en funciones con `@st.cache_data`, desde URL (el repositorio del curso, el propio repositorio en su dirección `raw`, un servicio WFS) o desde archivos del repositorio con rutas **relativas** a `app.py`.
- Los controles van en la barra lateral y el filtrado es pandas; la página se organiza con columnas o pestañas: una para la tabla, una para los gráficos, una para el mapa.
- El mapa se construye en una función con `@st.cache_resource` cuyos argumentos son los valores de los controles de los que depende, y se dibuja una copia (`copy.deepcopy()`), para que conserve la vista entre clics.
- Cada tabla, gráfico y mapa lleva título, y la página termina con las fuentes de los datos.

La aplicación siguiente reúne todo lo anterior y es la plantilla del proyecto. Es más larga que las otras, pero no tiene nada nuevo:

In [17]:
%%writefile apps/07-app-proyecto.py
# Aplicación de ejemplo del proyecto final: cantones de Costa Rica con su población.
# Ejecución: streamlit run apps/07-app-proyecto.py
import copy

import folium
import geopandas as gpd
import pandas as pd
import plotly.express as px
import streamlit as st
from shapely.geometry import Point
from streamlit_folium import st_folium

DATOS = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos"

# ---------- Configuración de la página (primera llamada a streamlit) ----------
st.set_page_config(page_title="Cantones de Costa Rica", page_icon="🗺️", layout="wide")


# ---------- Datos, en caché ----------
@st.cache_data
def cargar_cantones():
    cantones = gpd.read_file(f"{DATOS}/ign/cantones.gpkg").to_crs(4326)
    poblacion = pd.read_csv(f"{DATOS}/inec/cantones-2022.csv")
    return cantones.merge(poblacion[["codigo_canton", "poblacion", "area_km2", "densidad"]], on="codigo_canton", how="left")


@st.cache_data
def cargar_aerodromos():
    return gpd.read_file(f"{DATOS}/ign/aerodromos.gpkg").to_crs(4326)


def filtrar(provincia, poblacion_minima):
    cantones = cargar_cantones()
    seleccion = cantones if provincia == "Todas" else cantones[cantones["provincia"] == provincia]
    return seleccion[seleccion["poblacion"].fillna(0) >= poblacion_minima]


ETIQUETAS = {"poblacion": "Población (habitantes)", "densidad": "Densidad (hab/km²)", "area_km2": "Área (km²)"}


# El mapa depende de los filtros: un mapa en caché por cada combinación de valores (ver la sección de mapas)
@st.cache_resource
def crear_mapa(provincia, poblacion_minima, variable):
    seleccion = filtrar(provincia, poblacion_minima)
    aerodromos_seleccion = gpd.sjoin(cargar_aerodromos(), seleccion[["geometry"]], predicate="within")
    mapa = seleccion.explore(column=variable, cmap="YlOrRd", scheme="Quantiles", k=5, name="Cantones",
                             location=[9.75, -84.15], zoom_start=8,
                             tooltip=["canton", "provincia", variable], tiles="OpenStreetMap",
                             style_kwds={"color": "white", "weight": 0.5}, missing_kwds={"color": "#E6E6E6"},
                             legend_kwds={"caption": ETIQUETAS[variable] + ", 2022"})
    aerodromos_seleccion.explore(m=mapa, name="Aeródromos", color="#1F77B4", marker_kwds={"radius": 4}, tooltip=["nombre"])
    folium.LayerControl().add_to(mapa)
    return mapa


cantones = cargar_cantones()

# ---------- Controles ----------
st.sidebar.title("Filtros")
provincias = ["Todas"] + sorted(cantones["provincia"].unique())
provincia = st.sidebar.selectbox("Provincia", provincias)
poblacion_minima = st.sidebar.slider("Población mínima", 0, 100_000, 0, step=5_000)
variable = st.sidebar.selectbox("Variable del mapa y del gráfico", ["poblacion", "densidad", "area_km2"],
                                format_func=lambda v: {"poblacion": "Población", "densidad": "Densidad", "area_km2": "Área"}[v])

seleccion = filtrar(provincia, poblacion_minima)
aerodromos_seleccion = gpd.sjoin(cargar_aerodromos(), seleccion[["geometry"]], predicate="within")

# ---------- Página ----------
st.title("Cantones de Costa Rica: población y territorio, 2022")
st.markdown(f"**{provincia}** · {len(seleccion)} cantones · {aerodromos_seleccion.shape[0]} aeródromos")

col1, col2, col3 = st.columns(3)
col1.metric("Población", f"{seleccion['poblacion'].sum():,.0f}".replace(",", " "))
col2.metric("Área (km²)", f"{seleccion['area_km2'].sum():,.0f}".replace(",", " "))
col3.metric("Densidad (hab/km²)", f"{seleccion['poblacion'].sum() / seleccion['area_km2'].sum():,.0f}" if seleccion["area_km2"].sum() else "—")

pestana_tabla, pestana_grafico, pestana_mapa = st.tabs(["Tabla", "Gráfico", "Mapa"])

with pestana_tabla:
    tabla = seleccion.drop(columns="geometry").sort_values(variable, ascending=False)[["provincia", "canton", "poblacion", "area_km2", "densidad"]]
    st.dataframe(tabla, hide_index=True, width="stretch",
                 column_config={"provincia": "Provincia", "canton": "Cantón",
                                "poblacion": st.column_config.NumberColumn("Población", format="localized"),
                                "area_km2": st.column_config.NumberColumn("Área (km²)", format="%.1f"),
                                "densidad": st.column_config.NumberColumn("Densidad (hab/km²)", format="%.1f")})
    st.download_button("Descargar (CSV)", tabla.to_csv(index=False).encode("utf-8"), "cantones.csv", "text/csv")

with pestana_grafico:
    grafico = px.bar(tabla.head(15), x="canton", y=variable, labels={"canton": "Cantón", variable: ETIQUETAS[variable]},
                     title=f"{ETIQUETAS[variable]} de los cantones, 2022 ({provincia.lower() if provincia != 'Todas' else 'todas las provincias'})")
    st.plotly_chart(grafico, width="stretch")

with pestana_mapa:
    col_mapa, col_detalle = st.columns([3, 1])
    with col_mapa:
        mapa = copy.deepcopy(crear_mapa(provincia, poblacion_minima, variable))
        salida = st_folium(mapa, height=550, use_container_width=True, returned_objects=["last_clicked"])
    if salida and salida.get("last_clicked"):
        punto = Point(salida["last_clicked"]["lng"], salida["last_clicked"]["lat"])
        encontrado = cantones[cantones.contains(punto)]
        if not encontrado.empty:
            st.session_state["canton"] = encontrado.iloc[0]
    with col_detalle:
        st.subheader("Cantón")
        if "canton" in st.session_state:
            canton = st.session_state["canton"]
            st.write(f"**{canton['canton']}** ({canton['provincia']})")
            for columna in ["poblacion", "area_km2", "densidad"]:
                valor = canton[columna]
                formato = ",.0f" if columna == "poblacion" else ",.1f"
                st.write(f"{ETIQUETAS[columna]}: " + (f"{valor:{formato}}".replace(",", " ") if pd.notna(valor) else "sin datos"))
        else:
            st.info("Haga clic en un cantón.")

st.caption("Fuentes: Instituto Geográfico Nacional (SNIT) e Instituto Nacional de Estadística y Censos (2023). "
           "Datos preparados en el [repositorio del curso](https://github.com/gf0657-programacionsig/2026-ii/tree/main/datos).")

Writing apps/07-app-proyecto.py


In [18]:
app = probar("apps/07-app-proyecto.py")
app = probar("apps/07-app-proyecto.py", Provincia="Limón")

apps/07-app-proyecto.py: título: 'Cantones de Costa Rica: población y territorio, 2022' | controles: 3 | métricas: 3 | tablas: 1 | gráficos: 1 | mapas: 1 | errores: 0


apps/07-app-proyecto.py: título: 'Cantones de Costa Rica: población y territorio, 2022' | controles: 3 | métricas: 3 | tablas: 1 | gráficos: 1 | mapas: 1 | errores: 0


<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/v-desarrollo-aplicaciones-interactivas/img/streamlit-07-app-proyecto.png" alt="Aplicación completa con barra lateral de filtros, indicadores y la pestaña del mapa con los aeródromos y el detalle de un cantón" width="900">
  <figcaption><strong>Figura 7</strong>. La aplicación de ejemplo del proyecto, en disposición ancha: filtros en la barra lateral, indicadores, pestañas de tabla, gráfico y mapa, y el detalle del cantón elegido con un clic. Fuente: elaboración propia.</figcaption>
</figure>

Una aplicación puede tener **varias páginas** (una por sección del proyecto, por ejemplo) con `st.Page()` y `st.navigation()` en el archivo de entrada; la [documentación de aplicaciones multipágina](https://docs.streamlit.io/develop/concepts/multipage-apps/overview) lo explica en una página. Para el proyecto, una sola página con pestañas es suficiente y más simple de publicar.

*Ejercicios de esta sección: 10, en la sección de ejercicios al final del cuaderno.*

## Publicación en Streamlit Community Cloud

Una aplicación en `localhost` solo la ve quien la ejecuta. [Streamlit Community Cloud](https://streamlit.io/cloud) es el servicio gratuito de Snowflake que ejecuta aplicaciones de streamlit directamente desde un repositorio público de GitHub: cada vez que se hace un *push* al repositorio, la aplicación publicada se actualiza sola. Los pasos:

1. **El repositorio en GitHub**, público, con `app.py`, `requirements.txt` y `README.md` en la raíz (o en una carpeta, que se indica al publicar).
2. **`requirements.txt`**: una biblioteca por línea, con la versión fijada con `==`, que es la que se probó localmente (`pip list` o `conda list` la muestran). Community Cloud instala exactamente eso con `pip`; sin el archivo, solo instala streamlit, y la aplicación falla con `ModuleNotFoundError` en la primera importación. No se incluyen los módulos de la biblioteca estándar de Python (`os`, `math`, `json`). El archivo de la aplicación de ejemplo:

In [19]:
%%writefile apps/requirements.txt
streamlit==1.61.1
streamlit-folium==0.27.4
pandas==3.0.5
geopandas==1.1.4
folium==0.20.0
plotly==6.9.0
mapclassify==2.10.0

Writing apps/requirements.txt


3. **La cuenta**: en [share.streamlit.io](https://share.streamlit.io/) se inicia sesión con la cuenta de GitHub y se autoriza a Streamlit a leer los repositorios.
4. **Create app**: el botón de la esquina superior derecha del espacio de trabajo pide el repositorio, la rama, el archivo principal (`app.py`) y la dirección deseada, de la forma `https://<nombre>.streamlit.app`. *Deploy* instala las dependencias, ejecuta la aplicación y muestra el registro (*logs*) de la instalación mientras tanto: ahí aparecen los errores de `requirements.txt`.
5. **Después**: la dirección se agrega al `README.md` del repositorio y se entrega en Mediación Virtual. Desde el menú *Manage app*, en la esquina inferior derecha de la aplicación publicada, se ven los registros y se reinicia (*Reboot*) si algo cambió fuera del código. Un *push* al repositorio vuelve a desplegar la aplicación.

Dos límites del servicio gratuito que conviene conocer: los recursos de cada aplicación son modestos (memoria limitada; una capa de cientos de megabytes o un raster grande no caben: se simplifican o se recortan antes, como en el cuaderno de folium), y una aplicación sin visitas durante varios días se **duerme** y tarda un momento en despertar la primera vez que alguien la abre. Las alternativas con un flujo parecido son [Hugging Face Spaces](https://huggingface.co/spaces) y un servidor propio; el programa del curso admite "streamlit o un marco de trabajo similar", publicado en Internet.

*Ejercicios de esta sección: 11, en la sección de ejercicios al final del cuaderno.*

## Clínica de errores

Los errores propios de streamlit, provocados a propósito en aplicaciones mínimas y ejecutados con `AppTest`, que los reporta en lugar de detenerse. El primero es el más frecuente al construir controles en un ciclo:

In [20]:
%%writefile apps/clinica-1.py
# StreamlitDuplicateElementId: dos controles del mismo tipo con los mismos argumentos
# streamlit identifica cada control por su tipo y su etiqueta; dos iguales son ambiguos.
# Solución: etiquetas distintas o un argumento key único por control.
import streamlit as st

minimo = st.slider("Valor", 0, 100)
maximo = st.slider("Valor", 0, 100)

Writing apps/clinica-1.py


In [21]:
app = probar("apps/clinica-1.py")

apps/clinica-1.py: sin título | controles: 1 | métricas: 0 | tablas: 0 | gráficos: 0 | mapas: 0 | errores: 1
  ERROR: There are multiple `slider` elements with the same auto-generated ID. When this element is created, it is assigned an internal ID based on the element type and provided parameters. Multiple elements with the same type and parameters will cause this error.


In [22]:
%%writefile apps/clinica-2.py
# Un GeoDataFrame en st.dataframe(): la columna de geometría no se puede convertir al formato Arrow
# No hay error: streamlit la convierte a texto, muestra un aviso en la terminal y la tabla
# arrastra una columna enorme de coordenadas. Solución: drop(columns="geometry") antes de mostrar.
import geopandas as gpd
import streamlit as st

DATOS = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos"
provincias = gpd.read_file(f"{DATOS}/ign/provincias.gpkg")
st.dataframe(provincias)

Writing apps/clinica-2.py


In [23]:
app = probar("apps/clinica-2.py")
print(app.dataframe[0].value.dtypes)

apps/clinica-2.py: sin título | controles: 0 | métricas: 0 | tablas: 1 | gráficos: 0 | mapas: 0 | errores: 0
codigo_provincia     int32
provincia              str
geometry            string
dtype: object


In [24]:
%%writefile apps/clinica-3.py
# FileNotFoundError: una ruta de la computadora propia que no existe en el servidor
# La aplicación corre localmente si el archivo está ahí, pero en Community Cloud solo existe
# lo que está en el repositorio. Solución: una URL, o una ruta relativa a app.py de un archivo
# que sí esté en el repositorio.
import pandas as pd
import streamlit as st

cantones = pd.read_csv("C:/Users/estudiante/Descargas/cantones-2022.csv")
st.dataframe(cantones)

Writing apps/clinica-3.py


In [25]:
app = probar("apps/clinica-3.py")

apps/clinica-3.py: sin título | controles: 0 | métricas: 0 | tablas: 0 | gráficos: 0 | mapas: 0 | errores: 1
  ERROR: [Errno 2] No such file or directory: 'C:/Users/estudiante/Descargas/cantones-2022.csv'


In [26]:
%%writefile apps/clinica-4.py
# Un control que se usa antes de leerse: el programa se ejecuta de arriba abajo
# NameError: la variable provincia se usa en la línea 6 y el control que la define está en la 8.
import streamlit as st

st.title("Cantones")
st.write(f"Provincia elegida: {provincia}")
provincia = st.selectbox("Provincia", ["Alajuela", "Cartago", "Heredia"])

Writing apps/clinica-4.py


In [27]:
app = probar("apps/clinica-4.py")

apps/clinica-4.py: título: 'Cantones' | controles: 0 | métricas: 0 | tablas: 0 | gráficos: 0 | mapas: 0 | errores: 1
  ERROR: name 'provincia' is not defined


Otros dos errores no producen excepción. `use_container_width=True`, que aparece en muchos ejemplos y en el código que generan los asistentes, está obsoleto desde la versión 1.49: funciona todavía, con un aviso en la terminal, y se sustituye por `width="stretch"`. Y una capa en un SRC proyectado, como CRTM05, no da error en `st_folium()` ni en `st.map()`: el mapa aparece vacío o centrado en otro lugar, exactamente como en la clínica del cuaderno de folium; todo lo que va a un mapa se reproyecta antes a WGS84.

## Asistentes de IA para construir aplicaciones

streamlit es una de las bibliotecas sobre las que los asistentes generan código con más soltura, y también una de las que más cambia: en el código generado abundan funciones y argumentos de versiones anteriores (`st.cache`, `use_container_width`, `st.experimental_*`, `folium_static`). Las comprobaciones son cuatro: que la **versión** indicada en el prompt sea la instalada (`st.__version__`) y que las funciones existan en la [referencia de la API](https://docs.streamlit.io/develop/api-reference); que los **datos** se carguen desde una URL o una ruta relativa del repositorio, dentro de una función con `@st.cache_data`; que la aplicación **corra** con `streamlit run` sin errores en la terminal y que `AppTest` no reporte excepciones; y que `requirements.txt` incluya **todo** lo que la aplicación importa, con versiones. Un prompt de ejemplo:

> Actúe como asistente de desarrollo de aplicaciones web de ciencia de datos con streamlit, versión 1.61, y streamlit-folium **(rol)**. Tengo un GeoDataFrame `cantones` con 84 polígonos en WGS84 y las columnas `canton`, `provincia`, `poblacion`, `area_km2` y `densidad` (dos cantones sin datos), que se carga desde la URL `https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos/ign/cantones.gpkg` y se une por `codigo_canton` con el CSV `.../datos/inec/cantones-2022.csv` **(contexto)**. Escriba `app.py` con una lista desplegable de provincias en la barra lateral, tres indicadores (cantones, población y densidad de la selección), una tabla con formato de miles, un gráfico de barras de plotly de los diez cantones más poblados y un mapa de coropletas de la densidad hecho con `explore()` e incrustado con `st_folium()` **(tarea)**. Cargue los datos en una función con `@st.cache_data`, use `width="stretch"` en lugar de `use_container_width`, no use funciones obsoletas, agregue un comentario por sección y escriba también el `requirements.txt` con las versiones **(formato)**.

## Resumen

- Una **aplicación de streamlit** es un programa de Python que **se ejecuta completo, de arriba abajo, cada vez que la persona usuaria toca un control**. Las funciones `st.*` dibujan elementos en la página en el orden en que se llaman; se ejecuta con `streamlit run archivo.py` y se ve en el navegador en `localhost:8501`.
- Los **controles** (`st.selectbox()`, `st.multiselect()`, `st.slider()`, `st.checkbox()`, `st.radio()`, `st.text_input()`) dibujan un elemento de entrada y **retornan su valor**; el filtrado es pandas. Van en la **barra lateral** (`st.sidebar`). La página se organiza con `st.columns()`, `st.tabs()` y `st.expander()`; `st.session_state` recuerda valores entre ejecuciones.
- Las **salidas**: `st.write()` para todo, `st.title()`/`st.header()`/`st.markdown()` para el texto, `st.metric()` para indicadores, `st.dataframe()` para tablas interactivas (con `column_config` y sin la columna de geometría), `st.download_button()` para entregar archivos, `st.plotly_chart(fig, width="stretch")` para gráficos de plotly, `st.map()` para puntos y **`st_folium()`** (streamlit-folium) para los mapas de folium, leafmap y `explore()`, que además retorna los clics.
- Lo costoso se guarda en **caché**: las funciones que leen o calculan datos se decoran con `@st.cache_data`.
- La **aplicación del proyecto**: un repositorio con `app.py`, `requirements.txt` (con versiones) y `README.md`; `st.set_page_config()` primero, datos desde URL en caché, controles en la barra lateral, pestañas de tabla, gráfico y mapa, fuentes al pie.
- Se **publica** en Streamlit Community Cloud desde el repositorio de GitHub (*Create app* → *Deploy*), en una dirección `https://<nombre>.streamlit.app`; cada *push* la actualiza. Los recursos son limitados y las aplicaciones sin visitas se duermen.
- `AppTest` ejecuta una aplicación sin navegador, da acceso a sus elementos y simula la interacción: sirve para probar y para las celdas de verificación.

## Ejercicios

Los ejercicios se agrupan según la sección del cuaderno a la que corresponden; se recomienda realizarlos al concluir la sección respectiva. Cada aplicación se escribe en un archivo de la carpeta `apps/` con el nombre que indica el enunciado (con `%%writefile` en una celda de este cuaderno, o directamente en el editor) y se ejecuta con `streamlit run`. Varios ejercicios incluyen una **celda de verificación** que ejecuta la aplicación con `AppTest` y examina sus elementos: para que funcione, use los nombres de archivo y las etiquetas que indica el enunciado y ejecute la celda tal cual (en la versión publicada muestra un recordatorio, porque los archivos no existen). Las [soluciones de estos ejercicios](https://gf0657-programacionsig.github.io/2026-ii/soluciones-streamlit-aplicaciones-web) se publican después de la clase correspondiente.

### Introducción

1. Ejecute en la terminal `streamlit run apps/01-hola.py`, abra la aplicación en el navegador y active *Always rerun* en el menú de la esquina superior derecha. Edite `apps/01-hola.py` en el editor (cambie el título y agregue una línea con `st.write()`), guarde, y observe que la página cambia sola. Luego deténgala con Ctrl+C. Anote en una celda de texto qué dirección imprimió la terminal y qué ocurre en el navegador cuando la aplicación se detiene.

### Una aplicación es un programa

2. Escriba `apps/ejercicio-02.py`: una aplicación con el título "Calculadora de densidad", dos controles numéricos `st.number_input()` con las etiquetas "Población" y "Área (km²)" (con valores iniciales mayores que cero) y un `st.metric()` con la etiqueta "Densidad (hab/km²)" que muestre el cociente. Pruébela con `probar()` y en el navegador. ¿Qué pasa si el área es cero? Evítelo con una condición.

In [28]:
# Celda de verificación del ejercicio 2: ejecútela tal cual
try:
    app = AppTest.from_file(os.path.abspath("apps/ejercicio-02.py"), default_timeout=60).run()
    assert not app.exception, "La aplicación lanza una excepción: " + app.exception[0].value.splitlines()[0]
    assert app.title and app.title[0].value == "Calculadora de densidad", "El título debe ser 'Calculadora de densidad'"
    assert len(app.number_input) == 2, "Debe haber dos controles st.number_input()"
    assert app.metric and app.metric[0].label == "Densidad (hab/km²)", "Debe haber un st.metric() con la etiqueta 'Densidad (hab/km²)'"
    print("Ejercicio 2: ¡correcto!")
except FileNotFoundError:
    print("Resuelva el ejercicio (escriba apps/ejercicio-02.py) y vuelva a ejecutar esta celda.")

Resuelva el ejercicio (escriba apps/ejercicio-02.py) y vuelva a ejecutar esta celda.


### Datos y caché

3. Escriba `apps/ejercicio-03.py`: una aplicación que cargue `distritos-2022.csv` del INEC (en `datos/inec/` del repositorio del curso) en una función con `@st.cache_data`, muestre el título "Distritos de Costa Rica", tres indicadores (distritos, población total y promedio de ocupantes por vivienda) y la tabla de los 20 distritos más poblados con las columnas provincia, cantón, distrito y población, con encabezados en español y la población con separadores de miles. Agregue un botón para descargar esa tabla como CSV.

In [29]:
# Celda de verificación del ejercicio 3: ejecútela tal cual
try:
    app = AppTest.from_file(os.path.abspath("apps/ejercicio-03.py"), default_timeout=120).run()
    assert not app.exception, "La aplicación lanza una excepción: " + app.exception[0].value.splitlines()[0]
    assert len(app.metric) == 3, "Debe haber tres st.metric()"
    assert app.dataframe and len(app.dataframe[0].value) == 20, "La tabla debe tener 20 filas"
    assert app.get("download_button"), "Debe haber un st.download_button()"
    print("Ejercicio 3: ¡correcto!")
except FileNotFoundError:
    print("Resuelva el ejercicio (escriba apps/ejercicio-03.py) y vuelva a ejecutar esta celda.")

Resuelva el ejercicio (escriba apps/ejercicio-03.py) y vuelva a ejecutar esta celda.


4. Quite el decorador `@st.cache_data` de la función de carga del ejercicio anterior, ejecute la aplicación en el navegador y cambie varias veces de orden en la tabla o mueva un control. Mida con `time.time()` dentro de la función cuánto tarda la carga y muéstrelo con `st.caption()`. Vuelva a poner el decorador y compare. Explique en una celda de texto qué hace la caché y cuándo no debe usarse (pista: datos que cambian mientras la aplicación corre).

### Controles y disposición

5. Escriba `apps/ejercicio-05.py` a partir de `apps/03-controles.py`: agregue en la barra lateral un `st.multiselect()` con la etiqueta "Columnas" para elegir qué columnas de la tabla se muestran (todas al inicio) y un `st.slider()` con la etiqueta "Rango de densidad" que reciba dos valores (un rango, con `value=(mínimo, máximo)`) y filtre los cantones por densidad. Compruebe en el navegador que los filtros se combinan.

In [30]:
# Celda de verificación del ejercicio 5: ejecútela tal cual
try:
    app = AppTest.from_file(os.path.abspath("apps/ejercicio-05.py"), default_timeout=120).run()
    assert not app.exception, "La aplicación lanza una excepción: " + app.exception[0].value.splitlines()[0]
    etiquetas = [control.label for control in list(app.sidebar.multiselect) + list(app.sidebar.slider)]
    assert "Columnas" in etiquetas and "Rango de densidad" in etiquetas, "Faltan los controles 'Columnas' y 'Rango de densidad' en la barra lateral"
    rango = [control for control in app.sidebar.slider if control.label == "Rango de densidad"][0]
    assert isinstance(rango.value, (tuple, list)) and len(rango.value) == 2, "El control deslizante de densidad debe retornar un rango de dos valores"
    print("Ejercicio 5: ¡correcto!")
except FileNotFoundError:
    print("Resuelva el ejercicio (escriba apps/ejercicio-05.py) y vuelva a ejecutar esta celda.")

Resuelva el ejercicio (escriba apps/ejercicio-05.py) y vuelva a ejecutar esta celda.


6. **Prediga antes de ejecutar**: ¿qué muestra esta aplicación al abrirse, y qué muestra después de hacer clic tres veces en el botón? Escriba su predicción en una celda de texto, luego guárdela como `apps/ejercicio-06.py` y ejecútela en el navegador. Pista: el programa completo se ejecuta con cada clic, y solo `st.session_state` sobrevive.

```python
import streamlit as st

contador = 0
if "clics" not in st.session_state:
    st.session_state["clics"] = 0

if st.button("Contar"):
    contador = contador + 1
    st.session_state["clics"] = st.session_state["clics"] + 1

st.write(f"Variable: {contador} | Estado de sesión: {st.session_state['clics']}")
```

<details>
<summary>Explicación (ábrala después de predecir)</summary>

Al abrirse muestra `Variable: 0 | Estado de sesión: 0`. Tras tres clics muestra `Variable: 1 | Estado de sesión: 3`: en cada ejecución la variable `contador` vuelve a empezar en 0 y, como el botón retorna `True` solo en la ejecución que provocó el clic, llega a 1 y nada más; `st.session_state["clics"]` se crea una sola vez, porque la condición `not in` protege la inicialización, y conserva su valor entre ejecuciones. Es la diferencia entre una variable y el estado de sesión.

</details>

### Gráficos

7. Escriba `apps/ejercicio-07.py`: una aplicación con un `st.radio()` horizontal con la etiqueta "Tipo de gráfico" y las opciones "Barras" e "Histograma", y un `st.selectbox()` con la etiqueta "Variable" y las opciones población, área y densidad. Según el tipo, muestre con `st.plotly_chart()` un gráfico de barras de los 15 cantones con mayor valor de la variable o un histograma de la variable para los 84 cantones, con título y etiquetas de ejes con unidades, como en el cuaderno de plotly.

In [31]:
# Celda de verificación del ejercicio 7: ejecútela tal cual
try:
    app = AppTest.from_file(os.path.abspath("apps/ejercicio-07.py"), default_timeout=120).run()
    assert not app.exception, "La aplicación lanza una excepción: " + app.exception[0].value.splitlines()[0]
    assert app.radio and app.radio[0].label == "Tipo de gráfico", "Falta el st.radio() con la etiqueta 'Tipo de gráfico'"
    assert len(app.get("plotly_chart")) == 1, "Debe mostrarse exactamente un gráfico de plotly"
    app.radio[0].set_value("Histograma").run()
    assert not app.exception and len(app.get("plotly_chart")) == 1, "Con 'Histograma' también debe mostrarse un gráfico"
    print("Ejercicio 7: ¡correcto!")
except FileNotFoundError:
    print("Resuelva el ejercicio (escriba apps/ejercicio-07.py) y vuelva a ejecutar esta celda.")

Resuelva el ejercicio (escriba apps/ejercicio-07.py) y vuelva a ejecutar esta celda.


### Mapas

8. Escriba `apps/ejercicio-08.py` a partir de `apps/05-mapa-puntos.py`: agregue un `st.selectbox()` con la etiqueta "Provincia" que filtre los aeródromos por la provincia en que están (una unión espacial con los cantones o las provincias del IGN, como en el cuaderno de geopandas II, dentro de la función en caché) y muestre con `st.map()` solo los de la provincia elegida, con un `st.caption()` que diga cuántos son.

In [32]:
# Celda de verificación del ejercicio 8: ejecútela tal cual
try:
    app = AppTest.from_file(os.path.abspath("apps/ejercicio-08.py"), default_timeout=120).run()
    assert not app.exception, "La aplicación lanza una excepción: " + app.exception[0].value.splitlines()[0]
    assert app.selectbox and app.selectbox[0].label == "Provincia", "Falta el st.selectbox() con la etiqueta 'Provincia'"
    assert app.get("deck_gl_json_chart"), "Debe haber un st.map()"
    print("Ejercicio 8: ¡correcto!")
except FileNotFoundError:
    print("Resuelva el ejercicio (escriba apps/ejercicio-08.py) y vuelva a ejecutar esta celda.")

Resuelva el ejercicio (escriba apps/ejercicio-08.py) y vuelva a ejecutar esta celda.


9. Escriba `apps/ejercicio-09.py` a partir de `apps/06-mapa-folium.py`: sustituya el mapa de coropletas por uno de **dos capas** construido con folium o con `explore()` encadenado (`m=`): las provincias con un color por provincia y los aeródromos como marcadores circulares, con el control de capas y una etiqueta emergente en cada capa. Al hacer clic en el mapa, muestre a la derecha la provincia en que se hizo clic y cuántos aeródromos tiene (una unión espacial con los aeródromos). Compruébelo en el navegador con tres clics en provincias distintas.

In [33]:
# Celda de verificación del ejercicio 9: ejecútela tal cual
try:
    app = AppTest.from_file(os.path.abspath("apps/ejercicio-09.py"), default_timeout=120).run()
    assert not app.exception, "La aplicación lanza una excepción: " + app.exception[0].value.splitlines()[0]
    assert app.get("component_instance"), "Debe haber un mapa incrustado con st_folium()"
    print("Ejercicio 9: ¡correcto! (el clic en el mapa se comprueba en el navegador)")
except FileNotFoundError:
    print("Resuelva el ejercicio (escriba apps/ejercicio-09.py) y vuelva a ejecutar esta celda.")

Resuelva el ejercicio (escriba apps/ejercicio-09.py) y vuelva a ejecutar esta celda.


### La aplicación del proyecto y su publicación

10. Cree el repositorio del proyecto final en GitHub, con `app.py`, `requirements.txt` y `README.md`, y escriba en `app.py` la primera versión de la aplicación de su proyecto a partir de `apps/07-app-proyecto.py`: la configuración de la página, la carga en caché de las capas de su tarea 3 (desde URL), un control en la barra lateral, un indicador, la tabla y el mapa web. Pruébela con `streamlit run app.py` y con `AppTest`. Anote la dirección del repositorio en una celda de texto.

11. Publique la aplicación del ejercicio anterior en Streamlit Community Cloud (cree la cuenta con su cuenta de GitHub, *Create app*, *Deploy*) y anote su dirección en el `README.md` y en una celda de texto. Si la publicación falla, copie en una celda de texto las primeras líneas del error del registro y explique cuál fue la causa (casi siempre, una biblioteca que falta en `requirements.txt`).

### Asistentes

12. Use el prompt de ejemplo de la sección de asistentes, adaptado a las capas de su proyecto, y aplique las cuatro comprobaciones: versión y funciones existentes, datos desde URL en caché, ejecución sin errores (en la terminal y con `AppTest`) y `requirements.txt` completo. Escriba en una celda de texto qué tuvo que corregir del código generado y por qué.

## Referencias bibliográficas

Instituto Geográfico Nacional. (s. f.). *Sistema Nacional de Información Territorial (SNIT)* [Servicios WMS y WFS]. Registro Nacional de Costa Rica. Recuperado el 25 de octubre de 2026, de https://www.snitcr.go.cr/
\
\
Instituto Nacional de Estadística y Censos. (2023). *Resultados de la Estimación de Población y Vivienda 2022* [Conjunto de datos]. https://inec.cr/
\
\
Plotly Technologies Inc. (s. f.). *Plotly Open Source Graphing Library for Python*. Recuperado el 26 de setiembre de 2026, de https://plotly.com/python/
\
\
python-visualization. (s. f.). *Folium documentation* (versión 0.20.0). Recuperado el 25 de octubre de 2026, de https://python-visualization.github.io/folium/
\
\
Richards, T. (2023). *Streamlit for data science: Create interactive data apps in Python* (2.ª ed.). Packt Publishing.
\
\
Snowflake Inc. (s. f.-a). *Streamlit documentation* (versión 1.61). Recuperado el 1 de noviembre de 2026, de https://docs.streamlit.io/
\
\
Snowflake Inc. (s. f.-b). *Streamlit Community Cloud*. En *Streamlit documentation*. Recuperado el 1 de noviembre de 2026, de https://docs.streamlit.io/deploy/streamlit-community-cloud
\
\
Zwitch, R. (s. f.). *streamlit-folium* (versión 0.27) [Software]. Recuperado el 1 de noviembre de 2026, de https://github.com/randyzwitch/streamlit-folium